# Customer Support Ticket Classifier — Complete Pipeline

This notebook covers EDA, NLP preprocessing, model training, evaluation, and how to launch the Streamlit app.

**Targets**
- Priority classification (Low / Medium / High / Urgent)
- Category classification
- Resolution-time regression
- Product/subject nearest-neighbor match


In [ ]:
from pathlib import Path
import zipfile
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_absolute_error, r2_score
import nltk
from nltk.corpus import stopwords

ROOT = Path('.').resolve()
print('Working dir:', ROOT)


In [ ]:
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)
    words = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(words)


In [ ]:
zip_path = ROOT / 'customer_support_tickets_200k.zip'
csv_path = ROOT / 'customer_support_tickets_200k.csv'
if zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        with z.open('customer_support_tickets_200k.csv') as f:
            raw = pd.read_csv(f)
elif csv_path.exists():
    raw = pd.read_csv(csv_path)
else:
    raise FileNotFoundError('Dataset missing')
print(raw.shape)
raw.head(2)


In [ ]:
df = raw.copy()
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={
    'issue_description': 'text',
    'product': 'subject',
    'category': 'category',
    'priority': 'priority',
    'resolution_time_hours': 'resolution_time',
})
df = df[['text','subject','category','priority','resolution_time']].dropna()
df['priority'] = df['priority'].astype(str).str.strip().str.capitalize()
df = df[df['priority'].isin(['Low','Medium','High','Urgent'])]
counts = df['category'].value_counts()
df = df[df['category'].isin(counts[counts >= 100].index)].reset_index(drop=True)
df['cleaned'] = (df['subject'].astype(str) + ' ' + df['text'].astype(str)).apply(clean_text)
print(df.shape)
df['priority'].value_counts()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['priority'].value_counts().plot(kind='bar', ax=axes[0], title='Priority distribution')
df['category'].value_counts().head(10).plot(kind='bar', ax=axes[1], title='Top categories')
plt.tight_layout()
plt.show()


In [ ]:
le_p = LabelEncoder(); le_c = LabelEncoder()
df['priority_enc'] = le_p.fit_transform(df['priority'])
df['category_enc'] = le_c.fit_transform(df['category'])

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), stop_words='english', sublinear_tf=True, min_df=2)
X = tfidf.fit_transform(df['cleaned'])
X_train, X_test, y_train, y_test = train_test_split(
    X, df['priority_enc'], test_size=0.1, random_state=42, stratify=df['priority_enc']
)

model_p = LogisticRegression(max_iter=2000, class_weight='balanced', C=5, solver='lbfgs').fit(X_train, y_train)
pred = model_p.predict(X_test)
print('Priority accuracy:', round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=list(le_p.classes_), zero_division=0))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=le_p.classes_, yticklabels=le_p.classes_)
plt.title('Priority confusion matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()


In [ ]:
model_c = LogisticRegression(max_iter=2000, C=5, solver='lbfgs').fit(X, df['category_enc'])
model_t = LinearRegression().fit(X, df['resolution_time'])
time_pred = model_t.predict(X)
print('Resolution MAE:', round(mean_absolute_error(df['resolution_time'], time_pred), 3))
print('Resolution R2 :', round(r2_score(df['resolution_time'], time_pred), 3))
print('Category classes:', len(le_c.classes_))


## Launch the app

```bash
# Option A — train once, fast inference
python train_models.py
streamlit run app.py

# Option B — train at runtime
streamlit run ML_Final_project.py
```
